In [ ]:
# Lab type: extend
# Course: AI403 — Building Production AI Agents
# Lesson: Context Compaction and What Gets Lost Across Long Tasks
# Task: A long-running CRM clean-up agent compacts its context by summarising it. Extend the
# harness with pinned content, a harness-maintained ledger and offloading, then test it.

# Lab: Compaction Without Amnesia

**How these labs work.** The "model" here is `scripted_model`, a plain Python function that makes the same choices an LLM would in this situation. That keeps every run deterministic and free, so each bug reproduces exactly. The harness around it — the loop, the tools, the checks — is real code, and it's what you'll be fixing.

**Outputs are cleared.** Run every cell top to bottom.

## Setup

In [ ]:
# No installs needed: this lab uses the Python standard library only.

In [ ]:
import re

TASK = {"role": "user", "content": "Merge the 30 candidate duplicate pairs in the CRM. "
        "Constraint: never merge records that belong to different tenants."}
N_PAIRS = 30
CROSS_TENANT = {7, 14, 21, 28}          # pairs whose two records are in different tenants
DB = {"merges": []}

def tokens(msgs):
    return sum(len(m["content"]) // 4 for m in msgs)

def merge_pair(i):
    DB["merges"].append(i)
    history = "order;" * 330                          # the full record comes back (~700 tokens)
    return f'merged pair {i}: {{"id": "C-{1000 + i}", "name": "Acme Ltd", "history": "{history}"}}'

def skip_pair(i, reason):
    return f"skipped pair {i}: {reason}"

TOOLS = {"merge_pair": merge_pair, "skip_pair": skip_pair}

def scripted_model(messages):
    """Stand-in for an LLM: works through the pairs it can't see as finished.
    It reads 'merged pair N' / 'skipped pair N' anywhere in context, and ledger lines
    in the form 'merged: [1, 2]; skipped: [7]'. It skips cross-tenant pairs only if the
    tenant rule is in its context."""
    context = " ".join(m["content"] for m in messages)
    handled = {int(n) for n in re.findall(r"(?:merged|skipped) pair (\d+)", context)}
    for listed in re.findall(r"(?:merged|skipped): \[([\d, ]*)\]", context):
        handled |= {int(n) for n in listed.replace(" ", "").split(",") if n}
    knows_rule = "different tenants" in context
    for i in range(1, N_PAIRS + 1):
        if i in handled:
            continue
        if i in CROSS_TENANT and knows_rule:
            return {"type": "tool", "tool": "skip_pair", "args": {"i": i, "reason": "different tenants"}}
        return {"type": "tool", "tool": "merge_pair", "args": {"i": i}}
    return {"type": "final", "text": "All candidate pairs handled."}

def run(compact, tools=None, window=6000, max_steps=60):
    """The agent loop. The harness records every completed action in `ledger`."""
    tools = tools or TOOLS
    DB["merges"] = []
    msgs, ledger, compactions = [TASK], {"merged": [], "skipped": []}, 0
    for step in range(1, max_steps + 1):
        action = scripted_model(msgs)
        if action["type"] == "final":
            break
        result = tools[action["tool"]](**action["args"])
        ledger["merged" if action["tool"] == "merge_pair" else "skipped"].append(action["args"]["i"])
        msgs.append({"role": "tool", "content": result})
        if tokens(msgs) > window:
            msgs, compactions = compact(msgs, ledger), compactions + 1
    merges = DB["merges"]
    return {"steps": step, "compactions": compactions, "merges": len(merges),
            "duplicate_merges": len(merges) - len(set(merges)),
            "cross_tenant_merges": sorted(set(merges) & CROSS_TENANT)}

print(len(TOOLS), "tools;", N_PAIRS, "candidate pairs;", len(CROSS_TENANT), "are cross-tenant")

## The baseline: naive compaction

In [ ]:
def summarise(msgs):
    """Stand-in for an LLM-written summary: it records progress as a count."""
    text = " ".join(m["content"] for m in msgs)
    earlier = sum(int(n) for n in re.findall(r"(\d+) pairs merged", text))
    merged = earlier + sum(m["content"].startswith("merged pair") for m in msgs)
    return {"role": "user", "content": f"[Summary of earlier work] {merged} pairs merged so far."}

def naive_compact(msgs, ledger, keep_last=4):
    return [summarise(msgs[:-keep_last])] + msgs[-keep_last:]

print(run(naive_compact))

**Question 1.** Explain each bad number in the result: why did the run hit the 60-step cap, where did the duplicate merges come from, and why was pair 7 merged even though the rule was in the very first message?

In [ ]:
# Your explanation (as comments):

<details>
<summary>🔑 Reveal answer — Question 1</summary>

- **Duplicates and the step cap:** the summary records progress as a *count* ("24 pairs merged"), not *which* pairs. After each compaction the agent can only see the last four results, so it starts again from pair 1 — re-merging records it already merged, until the cap stops it. On a real CRM each of those is a duplicated side effect.
- **Pair 7:** before the first compaction the agent knew the rule and skipped pair 7 correctly. The summary then replaced everything but the last four messages — including the task message and its tenant constraint. Once the record of skipping pair 7 also scrolled out of view, the agent came back to pair 7 and, no longer knowing the rule, merged it across tenants.

Two of the four blind spots from the lesson: standing constraints and progress markers.

</details>

## Extension 1: pinned task and a harness ledger

**Question 2.** Write `pinned_compact(msgs, ledger, keep_last=4)` that returns: the original `TASK` (never summarised), one ledger message built **by code** from `ledger` in the form `[Ledger] merged: [...]; skipped: [...]`, and the last `keep_last` messages. Run it and compare.

In [ ]:
# Work here: pinned_compact

<details>
<summary>🔑 Reveal answer — Question 2</summary>

```python
def pinned_compact(msgs, ledger, keep_last=4):
    state = {"role": "user",
             "content": f"[Ledger] merged: {ledger['merged']}; skipped: {ledger['skipped']}"}
    recent = [m for m in msgs[-keep_last:] if m is not TASK]
    return [TASK, state] + recent

print(run(pinned_compact))
```

Same token threshold, but the run finishes in 31 steps with 26 merges, no duplicates and no cross-tenant merges: the rule is pinned, and the ledger comes from the tool results the harness already saw rather than from a paraphrase.

</details>

## Extension 2: offload large results

**Question 3.** Most of the context is 700-token merged records the agent never needs again. Write `merge_pair_offloaded(i)` that stores the full record in a `STORE` dict and returns a one-line reference (keep the text `merged pair {i}` so the model can still track progress). Run `run(pinned_compact, tools=...)` with it. How many compactions happen now?

In [ ]:
# Work here: merge_pair_offloaded

<details>
<summary>🔑 Reveal answer — Question 3</summary>

```python
STORE = {}

def merge_pair_offloaded(i):
    full = merge_pair(i)                       # performs the merge and returns the full record
    STORE[f"rec_{i}"] = full
    return f"merged pair {i} (full record saved as rec_{i}, {len(full) // 4} tokens)"

print(run(pinned_compact, tools={"merge_pair": merge_pair_offloaded, "skip_pair": skip_pair}))
```

No compactions at all: the whole task now fits under the threshold, and anything the agent needs later can be re-read from `STORE` by reference. Offloading avoids the loss that summarising causes.

</details>

## Extension 3: a compaction test

**Question 4.** Compaction bugs only appear on runs long enough to trigger compaction. Write `compaction_test(compact, **kwargs)` that runs the task with a *small* window (2,000 tokens, so compaction is guaranteed) and asserts: at least one compaction happened, no duplicate merges, no cross-tenant merges, and the run finished before the step cap. Show that it fails for `naive_compact` and passes for `pinned_compact`.

In [ ]:
# Work here: compaction_test

<details>
<summary>🔑 Reveal answer — Question 4</summary>

```python
def compaction_test(compact, **kwargs):
    out = run(compact, window=2000, **kwargs)
    problems = []
    if out["compactions"] < 1:
        problems.append("no compaction happened - the test proves nothing")
    if out["duplicate_merges"]:
        problems.append(f"{out['duplicate_merges']} duplicate merges")
    if out["cross_tenant_merges"]:
        problems.append(f"cross-tenant merges {out['cross_tenant_merges']}")
    if out["steps"] >= 60:
        problems.append("hit the step cap")
    return problems or ["pass"]

print("naive: ", compaction_test(naive_compact))
print("pinned:", compaction_test(pinned_compact))
```

The first check matters: a compaction test that never compacts passes trivially.

</details>

## Summary

1. Model-written summaries keep progress and drop standing _______.
2. The record of completed actions should be maintained by _______ from tool results.
3. Large results the agent may need again can be _______ behind a reference instead of summarised.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **constraints**
2. **the harness (code)**
3. **offloaded**

</details>